# Breast Cancer (VinDr-Mammo) — Qwen2.5-VL Models C

Pipeline:
1. Paths + load CSVs
2. Grouped labels (top findings + Other)
3. Train/val/test splits (VinDr official + carved val)
4. Box scaling helper (PNG-scaled, normalized 0–1000)
5. Build instruction JSONLs (plain + grounded)
9. Inference
10. Evaluation

## 1. Setup & paths

In [1]:
import os, json, ast, re, random, shutil
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
import torch

DATA_ROOT = Path(os.environ.get('VINDR_ROOT', 'data/vindr_mammo'))  # see README -> Data
RAW       = DATA_ROOT / 'vindr-mammo-a-large-scale-benchmark-dataset-for-computer-aided-detection-and-diagnosis-in-full-field-digital-mammography-1.0.0'
PNG       = DATA_ROOT / 'png'
ANN       = DATA_ROOT / 'annotations_processed'
LORA_DATA = DATA_ROOT / 'lora_data'
LORA_RUNS = DATA_ROOT / 'lora_runs'
YOLO_DIR  = DATA_ROOT / 'yolo'
RUNS      = DATA_ROOT / 'runs'

for d in [ANN, LORA_DATA, LORA_RUNS, YOLO_DIR, RUNS]:
    d.mkdir(parents=True, exist_ok=True)

BREAST_CSV  = RAW / 'breast-level_annotations.csv'
FINDING_CSV = RAW / 'finding_annotations.csv'

breast_df  = pd.read_csv(BREAST_CSV)
finding_df = pd.read_csv(FINDING_CSV)

# Drop the one corrupt image (failed DICOM->PNG conversion)
BAD_IDS = {'3e71b089f6e55d5791771b05f88518dd'}
breast_df  = breast_df[~breast_df['image_id'].isin(BAD_IDS)].reset_index(drop=True)
finding_df = finding_df[~finding_df['image_id'].isin(BAD_IDS)].reset_index(drop=True)

print('Breast rows:', len(breast_df), ' Finding rows:', len(finding_df))
print('PNG files:', len(list(PNG.glob('*.png'))))
print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU')

Breast rows: 19999  Finding rows: 20485
PNG files: 19999
CUDA: True | NVIDIA H200 NVL


## 2. Grouped labels

Multi-label, with rare findings collapsed into clinically meaningful groups:

| Group | Includes | Why |
|---|---|---|
| Mass | Mass | 1,123+ |
| Suspicious Calcification | Suspicious Calcification | 402+ |
| Asymmetry | Asymmetry, Focal Asymmetry, Global Asymmetry | clinically related, individually small |
| Architectural Distortion | Architectural Distortion | important cancer sign |
| Suspicious Lymph Node | Suspicious Lymph Node | clinically meaningful |
| Other Suspicious Signs | Skin Thickening, Skin Retraction, Nipple Retraction | too rare individually |

In [ ]:
GROUPS = {
    'Mass':                     ['Mass'],
    'Suspicious Calcification': ['Suspicious Calcification'],
    'Asymmetry':                ['Asymmetry', 'Focal Asymmetry', 'Global Asymmetry'],
    'Architectural Distortion': ['Architectural Distortion'],
    'Suspicious Lymph Node':    ['Suspicious Lymph Node'],
    'Other Suspicious Signs':   ['Skin Thickening', 'Skin Retraction', 'Nipple Retraction'],
}
GROUPED_CLASSES = list(GROUPS.keys())

GROUP_TO_PHRASE = {
    'Mass': 'A suspicious mass is present',
    'Suspicious Calcification': 'Suspicious calcifications are present',
    'Asymmetry': 'An asymmetry is present',
    'Architectural Distortion': 'Architectural distortion is present',
    'Suspicious Lymph Node': 'A suspicious lymph node is present',
    'Other Suspicious Signs': 'Other suspicious mammographic signs are present',
}

GROUP_TO_KNOWLEDGE = {
    'Mass': 'A mammographic mass is a space-occupying lesion with a visible shape, margin, and density.',
    'Suspicious Calcification': 'Suspicious calcifications appear as small high-density bright foci, often grouped or clustered.',
    'Asymmetry': 'Asymmetry is an area of denser breast tissue without a clear mass, possibly focal or global.',
    'Architectural Distortion': 'Architectural distortion appears as pulling or disruption of normal breast tissue without a definite mass.',
    'Suspicious Lymph Node': 'A suspicious lymph node appears enlarged, dense, rounded, or with abnormal morphology.',
    'Other Suspicious Signs': 'Other suspicious signs include skin thickening, nipple retraction, or skin retraction.',
}

def parse_finding_categories(x):
    if pd.isna(x): return []
    if isinstance(x, list): return x
    try:
        v = ast.literal_eval(str(x))
        return v if isinstance(v, list) else [str(v)]
    except Exception:
        return [str(x)]

def map_to_groups(categories):
    cats = parse_finding_categories(categories)
    if 'No Finding' in cats:
        return []
    out = set()
    for g, raws in GROUPS.items():
        for r in raws:
            if r in cats:
                out.add(g)
    return sorted(out)

# image_id -> set of grouped labels
image_to_groups = defaultdict(set)
for _, r in finding_df.iterrows():
    for g in map_to_groups(r['finding_categories']):
        image_to_groups[r['image_id']].add(g)
image_to_groups = {k: sorted(v) for k, v in image_to_groups.items()}

counts = Counter()
for groups in image_to_groups.values():
    for g in groups:
        counts[g] += 1

n_normal = len(breast_df) - sum(1 for img in breast_df['image_id'] if img in image_to_groups and len(image_to_groups[img]) > 0)
print(f'{"Normal (no finding)":30s} {n_normal}')
for g in GROUPED_CLASSES:
    print(f'{g:30s} {counts[g]}')

Normal (no finding)            18231
Mass                           1113
Suspicious Calcification       442
Asymmetry                      390
Architectural Distortion       119
Suspicious Lymph Node          53
Other Suspicious Signs         86


## 3. Splits

VinDr-Mammo provides an official `split` column (`training` / `test`). We carve a validation set out of the training studies, keeping all views of a study together (study-level split, not image-level).

In [ ]:
SEED = 0
VAL_FRAC = 0.10

train_studies_all = sorted(breast_df.loc[breast_df['split'] == 'training', 'study_id'].unique().tolist())
test_studies      = sorted(breast_df.loc[breast_df['split'] == 'test',     'study_id'].unique().tolist())

rng = random.Random(SEED)
rng.shuffle(train_studies_all)
n_val = int(len(train_studies_all) * VAL_FRAC)
val_studies   = sorted(train_studies_all[:n_val])
train_studies = sorted(train_studies_all[n_val:])

def studies_to_images(study_ids):
    return sorted(breast_df.loc[breast_df['study_id'].isin(study_ids), 'image_id'].unique().tolist())

train_ids = studies_to_images(train_studies)
val_ids   = studies_to_images(val_studies)
test_ids  = studies_to_images(test_studies)

print(f'Train studies: {len(train_studies):5d}  images: {len(train_ids):5d}')
print(f'Val   studies: {len(val_studies):5d}  images: {len(val_ids):5d}')
print(f'Test  studies: {len(test_studies):5d}  images: {len(test_ids):5d}')

assert not (set(train_studies) & set(val_studies))
assert not (set(train_studies) & set(test_studies))
assert not (set(val_studies)   & set(test_studies))

# save splits
(ANN / 'splits.json').write_text(json.dumps({
    'train_studies': train_studies, 'val_studies': val_studies, 'test_studies': test_studies,
    'train_ids': train_ids, 'val_ids': val_ids, 'test_ids': test_ids,
}))
print('Saved splits to', ANN / 'splits.json')

Train studies:  3600  images: 14399
Val   studies:   400  images:  1600
Test  studies:  1000  images:  4000
Saved splits to $VINDR_ROOT/annotations_processed/splits.json


## 4. Box scaling

Annotations store boxes in **original DICOM** coordinates. PNGs may have different dimensions. We scale to **PNG pixel coords**, then normalize to **0–1000** for the LLM target text. (Standard for Qwen2.5-VL grounding.)

In [ ]:
_png_dim_cache = {}

def get_png_size(image_id):
    if image_id in _png_dim_cache:
        return _png_dim_cache[image_id]
    p = PNG / f'{image_id}.png'
    if not p.exists():
        return None
    with Image.open(p) as im:
        _png_dim_cache[image_id] = im.size  # (W, H)
    return _png_dim_cache[image_id]

def box_to_norm1000(row, image_id):
    """Original DICOM coords -> PNG-scaled -> normalized 0-1000.
    Returns [x1, y1, x2, y2] ints, or None if invalid."""
    size = get_png_size(image_id)
    if size is None: return None
    png_w, png_h = size

    orig_w, orig_h = float(row['width']), float(row['height'])
    sx, sy = png_w / orig_w, png_h / orig_h

    x1 = float(row['xmin']) * sx
    y1 = float(row['ymin']) * sy
    x2 = float(row['xmax']) * sx
    y2 = float(row['ymax']) * sy

    x1 = max(0, min(png_w, x1)); x2 = max(0, min(png_w, x2))
    y1 = max(0, min(png_h, y1)); y2 = max(0, min(png_h, y2))
    if x2 - x1 < 2 or y2 - y1 < 2:
        return None

    return [
        int(round(x1 / png_w * 1000)),
        int(round(y1 / png_h * 1000)),
        int(round(x2 / png_w * 1000)),
        int(round(y2 / png_h * 1000)),
    ]

image_to_boxes = defaultdict(list)
for _, r in tqdm(finding_df.iterrows(), total=len(finding_df), desc='Scaling boxes'):
    if pd.isna(r.get('xmin', np.nan)): continue
    groups = map_to_groups(r['finding_categories'])
    if not groups: continue
    box = box_to_norm1000(r, r['image_id'])
    if box is None: continue
    for g in groups:
        image_to_boxes[r['image_id']].append({'group': g, 'box': box})

print(f'Images with at least one grouped box: {len(image_to_boxes)}')

sample_id = next(iter(image_to_boxes))
print(f'\nSample {sample_id}: {image_to_boxes[sample_id]}')

Images with at least one grouped box: 1768

Sample <image_id>: [{'group': 'Mass', 'box': [841, 492, 887, 527]}]


## 5. Build instruction JSONLs


In [15]:
PLAIN_DIR = LORA_DATA / 'plain_grouped'
GND_DIR   = LORA_DATA / 'grounded_grouped'
PLAIN_DIR.mkdir(parents=True, exist_ok=True)
GND_DIR.mkdir(parents=True, exist_ok=True)

SYSTEM_MAMMO = (
    'You are an expert radiology assistant for mammography. '
    'Answer carefully using only the image information.'
)

def get_breast_row(image_id):
    rows = breast_df[breast_df['image_id'] == image_id]
    return rows.iloc[0] if len(rows) else None

def make_messages(image_path, user_text, assistant_text):
    return [
        {'role': 'system',    'content': [{'type': 'text', 'text': SYSTEM_MAMMO}]},
        {'role': 'user',      'content': [{'type': 'image', 'image': str(image_path)},
                                          {'type': 'text', 'text': user_text}]},
        {'role': 'assistant', 'content': [{'type': 'text', 'text': assistant_text}]},
    ]

def finding_classification_answer(groups):
    if not groups: return 'No suspicious mammographic finding is present.'
    return 'The suspicious mammographic findings present are: ' + ', '.join(groups) + '.'

def birads_density_answer(row):
    return f'The BI-RADS category is {row["breast_birads"]}. The breast density is {row["breast_density"]}.'

def plain_report_answer(groups, row):
    parts = [birads_density_answer(row)]
    if not groups:
        parts.append('No suspicious mammographic finding is present.')
    else:
        for g in groups:
            parts.append(GROUP_TO_PHRASE[g] + '.')
    return ' '.join(parts)

def grounded_report_answer(image_id, groups, row):
    parts = [birads_density_answer(row)]
    items = image_to_boxes.get(image_id, [])
    if not groups or not items:
        parts.append('No suspicious mammographic finding is present.')
        return ' '.join(parts)
    for it in items:
        b = it['box']
        parts.append(f'{GROUP_TO_PHRASE[it["group"]]} <box>{b[0]} {b[1]} {b[2]} {b[3]}</box>.')
    return ' '.join(parts)

def build_examples_for_image(image_id, grounded=False):
    png_path = PNG / f'{image_id}.png'
    if not png_path.exists(): return []
    row = get_breast_row(image_id)
    if row is None: return []

    groups = image_to_groups.get(image_id, [])
    examples = []

    # Task 1: finding classification
    examples.append({
        'image': str(png_path), 'task': 'finding_classification',
        'messages': make_messages(png_path,
            'What suspicious mammographic findings are present?',
            finding_classification_answer(groups))
    })

    # Task 2: BI-RADS + density
    examples.append({
        'image': str(png_path), 'task': 'birads_density',
        'messages': make_messages(png_path,
            'What is the BI-RADS category and breast density?',
            birads_density_answer(row))
    })

    # Task 3: report
    if grounded:
        report = grounded_report_answer(image_id, groups, row)
        user = ('Generate a grounded mammography report. For each suspicious finding, '
                'append its location as <box>x_min y_min x_max y_max</box> in normalized 0-1000 coordinates.')
        task = 'grounded_report'
    else:
        report = plain_report_answer(groups, row)
        user = 'Generate a concise mammography finding report.'
        task = 'plain_report'
    examples.append({'image': str(png_path), 'task': task,
                     'messages': make_messages(png_path, user, report)})

    # Task 4: per-finding grounding (grounded only)
    if grounded:
        for it in image_to_boxes.get(image_id, []):
            g, b = it['group'], it['box']
            examples.append({
                'image': str(png_path), 'task': 'single_finding_grounding',
                'messages': make_messages(png_path,
                    f'Locate {g}. {GROUP_TO_KNOWLEDGE[g]}',
                    f'The {g} is located at <box>{b[0]} {b[1]} {b[2]} {b[3]}</box>.')
            })

    return examples

def write_split(ids, split_name, out_dir, grounded):
    out_path = out_dir / f'{split_name}.jsonl'
    n_img = n_ex = 0
    with open(out_path, 'w') as f:
        for image_id in tqdm(sorted(ids), desc=f'{out_dir.name}/{split_name}'):
            exs = build_examples_for_image(image_id, grounded=grounded)
            if not exs: continue
            n_img += 1
            for ex in exs:
                f.write(json.dumps(ex) + '\n')
                n_ex += 1
    print(f'  {split_name}: images={n_img}, examples={n_ex}')

print('=== Plain grouped (Model A) ===')
for split, ids in [('train', train_ids), ('val', val_ids), ('test', test_ids)]:
    write_split(ids, split, PLAIN_DIR, grounded=False)

print('\n=== Grounded grouped (Model B) ===')
for split, ids in [('train', train_ids), ('val', val_ids), ('test', test_ids)]:
    write_split(ids, split, GND_DIR, grounded=True)

=== Plain grouped (Model A) ===


## 5.1 Data enriched on rare findings

building rare-enriched training set for model C
- cap normal images at 4,000
- repeat finding-positive images by class frequency
- val and test sets unchanged


In [ ]:
import random
from collections import Counter
from tqdm.auto import tqdm

NORMAL_CAP = 4000
SEED = 0

REPEAT_FACTORS = {
    'Mass':                     2,
    'Suspicious Calcification': 2,
    'Asymmetry':                4,
    'Architectural Distortion': 10,
    'Suspicious Lymph Node':    15,
    'Other Suspicious Signs':   10,
}

rng = random.Random(SEED)

# split training images into normal vs has-finding
train_image_set = set(train_ids)
normal_ids  = [i for i in train_image_set if not image_to_groups.get(i)]
finding_ids = [i for i in train_image_set if image_to_groups.get(i)]

print(f'Total train images:         {len(train_image_set)}')
print(f'  Normal:                   {len(normal_ids)}')
print(f'  With finding:             {len(finding_ids)}')

# cap normals
rng.shuffle(normal_ids)
sampled_normal = normal_ids[:NORMAL_CAP]
print(f'\nNormal cap → kept {len(sampled_normal)} of {len(normal_ids)}')

# compute per-image repeat counts
def image_repeat_count(image_id):
    groups = image_to_groups.get(image_id, [])
    if not groups:
        return 1
    return max(REPEAT_FACTORS.get(g, 1) for g in groups)

train_list = list(sampled_normal)
per_class_examples = Counter()

for img_id in finding_ids:
    n = image_repeat_count(img_id)
    train_list.extend([img_id] * n)
    for g in image_to_groups.get(img_id, []):
        per_class_examples[g] += n

rng.shuffle(train_list)

# summary
print(f'\nFinal train list (image-instances): {len(train_list)}\n')
print(f'{"Class":30s} {"orig":>6s} {"repeated":>10s} {"% of train":>12s}')
print('-' * 64)
for g in GROUPED_CLASSES:
    orig = sum(1 for i in finding_ids if g in image_to_groups.get(i, []))
    pct  = 100 * per_class_examples[g] / len(train_list)
    print(f'{g:30s} {orig:>6d} {per_class_examples[g]:>10d} {pct:>11.2f}%')
norm_pct = 100 * len(sampled_normal) / len(train_list)
print(f'{"Normal (No Finding)":30s} {len(normal_ids):>6d} {len(sampled_normal):>10d} {norm_pct:>11.2f}%')

ENRICHED_DIR = LORA_DATA / 'grounded_rare_enriched'
ENRICHED_DIR.mkdir(parents=True, exist_ok=True)

# train
n_train_ex = 0
with open(ENRICHED_DIR / 'train.jsonl', 'w') as f:
    for image_id in tqdm(train_list, desc='train'):
        exs = build_examples_for_image(image_id, grounded=True)
        for ex in exs:
            f.write(json.dumps(ex) + '\n')
            n_train_ex += 1

# val + test
for split_name, ids in [('val', val_ids), ('test', test_ids)]:
    n_img = n_ex = 0
    with open(ENRICHED_DIR / f'{split_name}.jsonl', 'w') as f:
        for image_id in tqdm(sorted(ids), desc=split_name):
            exs = build_examples_for_image(image_id, grounded=True)
            if not exs: continue
            n_img += 1
            for ex in exs:
                f.write(json.dumps(ex) + '\n')
                n_ex += 1
    print(f'{split_name}: {n_img} images → {n_ex} examples')

print(f'\nTrain: {len(train_list)} image-instances → {n_train_ex} examples')
print(f'\nDone. Path: {ENRICHED_DIR}')

Total train images:         14399
  Normal:                   13130
  With finding:             1269

Normal cap → kept 4000 of 13130

Final train list (image-instances): 8671

Class                            orig   repeated   % of train
----------------------------------------------------------------
Mass                              808       2437       28.11%
Suspicious Calcification          296       1197       13.80%
Asymmetry                         288       1324       15.27%
Architectural Distortion           86        860        9.92%
Suspicious Lymph Node              37        555        6.40%
Other Suspicious Signs             66        700        8.07%
Normal (No Finding)             13130       4000       46.13%


val: 1600 images → 5022 examples


test: 4000 images → 12482 examples

Train: 8671 image-instances → 33746 examples

Done. Path: $VINDR_ROOT/lora_data/grounded_rare_enriched


### 5.1a Module B+ or c

In [ ]:
# ============================================================
# model C — continued fine-tuning of model B on rare-enriched data
# ============================================================

import torch, json, gc
from pathlib import Path
from datasets import Dataset
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
from qwen_vl_utils import process_vision_info
from peft import PeftModel
from trl import SFTConfig, SFTTrainer

# clear GPU memory first
try:
    del trainer
except NameError: pass
try:
    del model
except NameError: pass
try:
    del base
except NameError: pass
gc.collect()
torch.cuda.empty_cache()
print('GPU memory cleared.')

# ---- Paths ----
MODEL_ID        = 'Qwen/Qwen2.5-VL-7B-Instruct'
MODEL_B_ADAPTER = LORA_RUNS / 'qwen25vl7b_grounded_grouped'
DATA_DIR        = LORA_DATA / 'grounded_rare_enriched'
OUT_DIR         = LORA_RUNS / 'qwen25vl7b_model_c'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- load model B and make its adapter trainable ----
processor = AutoProcessor.from_pretrained(MODEL_ID)
base = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto'
)
model = PeftModel.from_pretrained(
    base,
    str(MODEL_B_ADAPTER),
    is_trainable=True,        # <-- key flag
)
model.config.use_cache = False
print('Loaded Model B as trainable starting point.')

# ---- load rare-enriched data ----
def load_jsonl(path):
    return Dataset.from_list([json.loads(l) for l in open(path)])

train_ds = load_jsonl(DATA_DIR / 'train.jsonl')
val_ds   = load_jsonl(DATA_DIR / 'val.jsonl')
print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')

# ---- collator (same as before) ----
def clean_messages(messages):
    out = []
    for m in messages:
        new_content = [{k: v for k, v in c.items() if v is not None} for c in m['content']]
        out.append({'role': m['role'], 'content': new_content})
    return out

def collate_qwen(examples):
    messages = [clean_messages(ex['messages']) for ex in examples]
    texts = [processor.apply_chat_template(m, tokenize=False, add_generation_prompt=False) for m in messages]
    image_inputs, video_inputs = process_vision_info(messages)
    batch = processor(text=texts, images=image_inputs, videos=video_inputs,
                      padding=True, return_tensors='pt')
    labels = batch['input_ids'].clone()
    pad_id = processor.tokenizer.pad_token_id
    if pad_id is not None:
        labels[labels == pad_id] = -100
    image_token_id = processor.tokenizer.convert_tokens_to_ids('<|image_pad|>')
    if image_token_id is not None:
        labels[labels == image_token_id] = -100
    batch['labels'] = labels
    return batch

# ---- training config — LOWER LR, FEWER EPOCHS ----
cfg = SFTConfig(
    output_dir=str(OUT_DIR),
    num_train_epochs=1,                 # only 1 epoch for continued fine-tuning
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-5,                 # 10x lower than initial 1e-4
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    logging_steps=10,
    save_strategy='epoch',
    eval_strategy='epoch',
    bf16=True,
    gradient_checkpointing=True,
    remove_unused_columns=False,
    report_to='none',
    dataset_kwargs={'skip_prepare_dataset': True},
)

trainer = SFTTrainer(
    model=model, args=cfg,
    train_dataset=train_ds, eval_dataset=val_ds,
    data_collator=collate_qwen,
    processing_class=processor,
)

trainer.train()
trainer.save_model(str(OUT_DIR))
print(f'\nModel C saved to: {OUT_DIR}')

GPU memory cleared.


Loaded Model B as trainable starting point.


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Train: 33746 | Val: 5022


[RANK 0] Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 151645, 'bos_token_id': None, 'pad_token_id': 151643}.



Model C saved to: $VINDR_ROOT/lora_runs/qwen25vl7b_model_c


## 9. Inference

run on test set. use the same code for both Model A and Model B — just point `ADAPTER_DIR` at the right run.

In [ ]:
from peft import PeftModel
EVAL_TIER = 'model_c'
ADAPTER_DIR = LORA_RUNS / 'qwen25vl7b_model_c'
EVAL_DATA = LORA_DATA / 'grounded_grouped' / 'test.jsonl'   # original test, not enriched
PRED_OUT = LORA_RUNS / 'preds_model_c.jsonl'

processor = AutoProcessor.from_pretrained(MODEL_ID)
base = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto')
model = PeftModel.from_pretrained(base, str(ADAPTER_DIR))
model.eval()
print('Loaded adapter:', ADAPTER_DIR)

Loaded adapter: $VINDR_ROOT/lora_runs/qwen25vl7b_model_c


In [ ]:
@torch.no_grad()
def generate_one(messages, max_new_tokens=256):
    inf_messages = [m for m in messages if m['role'] != 'assistant']
    text = processor.apply_chat_template(inf_messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(inf_messages)
    inputs = processor(text=[text], images=image_inputs, videos=video_inputs,
                       padding=True, return_tensors='pt').to(model.device)
    gen = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    trimmed = gen[:, inputs['input_ids'].shape[1]:]
    return processor.batch_decode(trimmed, skip_special_tokens=True)[0].strip()

test_examples = [json.loads(l) for l in open(EVAL_DATA)]
print(f'{len(test_examples)} test examples')

with open(PRED_OUT, 'w') as f:
    for ex in tqdm(test_examples, desc='Generating'):
        pred = generate_one(ex['messages'])
        gt = ex['messages'][-1]['content'][0]['text']
        out = {
            'image': ex['image'],
            'task': ex['task'],
            'gt': gt,
            'pred': pred,
        }
        f.write(json.dumps(out) + '\n')

print('Saved predictions to', PRED_OUT)

12482 test examples


Saved predictions to $VINDR_ROOT/lora_runs/preds_model_c.jsonl


## 10. Evaluation


In [5]:
from sklearn.metrics import f1_score, precision_recall_fscore_support, accuracy_score, cohen_kappa_score

preds = [json.loads(l) for l in open(PRED_OUT)]
by_task = defaultdict(list)
for p in preds: by_task[p['task']].append(p)
for t, items in by_task.items():
    print(f'{t:30s} {len(items)}')

# ---- helpers ----
GROUP_KEYWORDS = {
    'Mass': ['mass'],
    'Suspicious Calcification': ['calcification'],
    'Asymmetry': ['asymmetry'],
    'Architectural Distortion': ['architectural distortion', 'distortion'],
    'Suspicious Lymph Node': ['lymph node'],
    'Other Suspicious Signs': ['other suspicious', 'skin thickening', 'skin retraction', 'nipple retraction'],
}

def extract_findings(text):
    t = text.lower()
    out = set()
    for g, kws in GROUP_KEYWORDS.items():
        if any(k in t for k in kws):
            out.add(g)
    return out

def to_binary_vec(groups):
    return [1 if g in groups else 0 for g in GROUPED_CLASSES]

BIRADS_RE  = re.compile(r'bi-?rads\s*(?:category\s*)?([1-5])', re.I)
DENSITY_RE = re.compile(r'density\s*([abcd])', re.I)
BOX_RE     = re.compile(r'<box>\s*(\d+)\s+(\d+)\s+(\d+)\s+(\d+)\s*</box>')

def extract_birads(text):
    m = BIRADS_RE.search(text)
    return m.group(1) if m else None

def extract_density(text):
    m = DENSITY_RE.search(text)
    return m.group(1).upper() if m else None

def extract_boxes(text):
    return [list(map(int, m)) for m in BOX_RE.findall(text)]

def iou(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    iw, ih = max(0, ix2 - ix1), max(0, iy2 - iy1)
    inter = iw * ih
    ua = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / ua if ua > 0 else 0.0

finding_classification         4000
birads_density                 4000
grounded_report                4000
single_finding_grounding       482


In [ ]:
# finding classification metrics
if 'finding_classification' in by_task:
    items = by_task['finding_classification']
    y_true, y_pred = [], []
    for it in items:
        y_true.append(to_binary_vec(extract_findings(it['gt'])))
        y_pred.append(to_binary_vec(extract_findings(it['pred'])))
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    micro_f1 = f1_score(y_true, y_pred, average='micro', zero_division=0)
    print(f'\nFinding classification: macro F1 = {macro_f1:.3f}, micro F1 = {micro_f1:.3f}')
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average=None, zero_division=0, labels=list(range(len(GROUPED_CLASSES))))
    for i, g in enumerate(GROUPED_CLASSES):
        print(f'  {g:30s} P={p[i]:.3f}  R={r[i]:.3f}  F1={f[i]:.3f}')

# BI-RADS + density
if 'birads_density' in by_task:
    items = by_task['birads_density']
    bg, bp, dg, dp = [], [], [], []
    for it in items:
        b_g, b_p = extract_birads(it['gt']), extract_birads(it['pred'])
        d_g, d_p = extract_density(it['gt']), extract_density(it['pred'])
        if b_g and b_p: bg.append(int(b_g)); bp.append(int(b_p))
        if d_g and d_p: dg.append(d_g); dp.append(d_p)
    print(f'\nBI-RADS:  acc={accuracy_score(bg, bp):.3f}, weighted kappa={cohen_kappa_score(bg, bp, weights="quadratic"):.3f}  ({len(bg)} valid)')
    print(f'Density:  acc={accuracy_score(dg, dp):.3f}, weighted kappa={cohen_kappa_score(dg, dp, weights="quadratic"):.3f}  ({len(dg)} valid)')

# grounding IoU (Model B / grounded tier only)
if 'single_finding_grounding' in by_task:
    items = by_task['single_finding_grounding']
    ious = []
    for it in items:
        gt_b = extract_boxes(it['gt'])
        pr_b = extract_boxes(it['pred'])
        if not gt_b or not pr_b:
            ious.append(0.0); continue
        ious.append(iou(gt_b[0], pr_b[0]))
    ious = np.array(ious)
    print(f'\nGrounding: mean IoU = {ious.mean():.3f}, IoU@0.5 = {(ious >= 0.5).mean():.3f}  ({len(ious)} cases)')

# report extraction (findings from generated report)
for task_name in ['plain_report', 'grounded_report']:
    if task_name not in by_task: continue
    items = by_task[task_name]
    y_true, y_pred = [], []
    for it in items:
        y_true.append(to_binary_vec(extract_findings(it['gt'])))
        y_pred.append(to_binary_vec(extract_findings(it['pred'])))
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    print(f'\n{task_name} (finding extraction from report): macro F1 = {f1_score(y_true, y_pred, average="macro", zero_division=0):.3f}')


Finding classification: macro F1 = 0.275, micro F1 = 0.352
  Mass                           P=0.381  R=0.507  F1=0.435
  Suspicious Calcification       P=0.507  R=0.667  F1=0.576
  Asymmetry                      P=0.107  R=0.231  F1=0.146
  Architectural Distortion       P=0.041  R=0.083  F1=0.055
  Suspicious Lymph Node          P=0.267  R=0.400  F1=0.320
  Other Suspicious Signs         P=0.069  R=0.429  F1=0.119

BI-RADS:  acc=0.733, weighted kappa=0.549  (4000 valid)
Density:  acc=0.839, weighted kappa=0.597  (4000 valid)

Grounding: mean IoU = 0.368, IoU@0.5 = 0.434  (482 cases)

grounded_report (finding extraction from report): macro F1 = 0.289
